# Finviz intraday patterns & interactions

Este notebook está pensado para funcionar con *cualquier archivo similar* (CSV o TSV) con columnas:

`timestamp, category, ticker, price, change_pct, volume`

Secciones clave

- Carga y limpieza robusta
- Evolución temporal por categorías
- Transiciones de un ticker entre categorías (interacciones)
- Picos de volumen y su contexto
- Conclusiones accionables


In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from utils import load_intraday_file, compute_transitions, top_volume_spikes

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120


In [ ]:
# Config: cambia SOLO este path si quieres analizar otro fichero
# - Soporta CSV (coma) y TSV (tab)
# - Si arrastras otro archivo similar al workspace, pon su nombre aquí

DATA_PATH = 'finviz_snapshots.csv'


In [ ]:
# Load data
finviz_df = load_intraday_file(DATA_PATH)

# Derived time buckets
finviz_df['minute'] = finviz_df['timestamp'].dt.floor('min')
finviz_df['hour'] = finviz_df['timestamp'].dt.hour

print(finviz_df.shape)
finviz_df.head(10)


In [ ]:
# Quick sanity checks
print(finviz_df['timestamp'].min())
print(finviz_df['timestamp'].max())
print(finviz_df['category'].nunique())
print(finviz_df['ticker'].nunique())

finviz_df[['price','change_pct','volume']].describe().T


In [ ]:
# Intraday activity: how much is happening each minute?
minute_summary = (finviz_df.groupby('minute')
                  .agg(n_rows=('ticker','size'),
                       n_tickers=('ticker','nunique'),
                       total_volume=('volume','sum'),
                       mean_change=('change_pct','mean'))
                  .reset_index()
                  .sort_values('minute'))

fig, ax1 = plt.subplots(figsize=(11, 4.6))
ax1.plot(minute_summary['minute'], minute_summary['total_volume'], color='#1f77b4', linewidth=1.8)
ax1.set_ylabel('Total volume (sum)')
ax1.set_title('Actividad intradía por minuto: volumen total y cambio medio')
ax1.tick_params(axis='x', rotation=30)

ax2 = ax1.twinx()
ax2.plot(minute_summary['minute'], minute_summary['mean_change'] * 100.0, color='#ff7f0e', linewidth=1.4, alpha=0.9)
ax2.set_ylabel('Mean change_pct (%)')

fig.tight_layout()
plt.show()


In [ ]:
# Category footprint over time (top categories)
cat_counts = finviz_df['category'].value_counts()
top_cats = cat_counts.head(10).index.tolist()

cat_minute_counts = (finviz_df[finviz_df['category'].isin(top_cats)]
                     .groupby(['minute', 'category'])['ticker']
                     .nunique()
                     .reset_index(name='n_tickers'))

cat_pivot = cat_minute_counts.pivot(index='minute', columns='category', values='n_tickers').fillna(0).sort_index()

plt.figure(figsize=(11, 5.0))
for c in cat_pivot.columns:
    plt.plot(cat_pivot.index, cat_pivot[c], linewidth=1.2, alpha=0.9, label=c)
plt.title('Top categorías a lo largo del tiempo (tickers únicos por minuto)')
plt.ylabel('Unique tickers')
plt.xlabel('Minute (UTC)')
plt.xticks(rotation=30)
plt.legend(ncols=2, fontsize=8, frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
# Ticker transitions between categories (interactions)
transitions = compute_transitions(finviz_df)
print(transitions.shape)
transitions.head(10)


In [ ]:
# Most common transitions (from -> to)
trans_counts = (transitions.groupby(['from_category','to_category'])
                .size()
                .reset_index(name='count')
                .sort_values('count', ascending=False))

trans_counts.head(20)


In [ ]:
# Heatmap of transitions for the most relevant categories
cat_rank = pd.concat([trans_counts['from_category'], trans_counts['to_category']]).value_counts()
keep_cats = cat_rank.head(15).index.tolist()

heat_df = (trans_counts.pivot(index='from_category', columns='to_category', values='count')
           .reindex(index=keep_cats, columns=keep_cats)
           .fillna(0))

plt.figure(figsize=(10, 7))
im = plt.imshow(heat_df.values, aspect='auto', cmap='viridis')
plt.colorbar(im, fraction=0.046, pad=0.04, label='Transition count')
plt.xticks(range(len(heat_df.columns)), heat_df.columns, rotation=45, ha='right')
plt.yticks(range(len(heat_df.index)), heat_df.index)
plt.title('Transiciones de categoría (top 15 categorías)')
plt.tight_layout()
plt.show()


In [ ]:
# Which tickers are the "most mobile" across categories?
# Mobility = number of category changes observed
mobility = (transitions.groupby('ticker')
            .size()
            .reset_index(name='n_transitions')
            .sort_values('n_transitions', ascending=False))

mobility.head(20)


In [ ]:
# Volume spikes (ticker-specific) and their category context
spikes = top_volume_spikes(finviz_df, q=0.95, min_volume=1.0)
print(spikes.shape)

spikes[['timestamp','ticker','category','volume','price','change_pct']].head(20)


In [ ]:
# Plot volume spikes timeline (top N spikes)
if len(spikes) > 0:
    topN = 30
    sp_top = spikes.head(topN).copy().sort_values('timestamp')

    plt.figure(figsize=(11, 4.8))
    plt.scatter(sp_top['timestamp'], sp_top['volume'], s=60, alpha=0.85, color='#d62728')
    for _, r in sp_top.iterrows():
        plt.text(r['timestamp'], r['volume'], r['ticker'], fontsize=8, rotation=25, ha='left', va='bottom')
    plt.title('Top picos de volumen (ticker-specific, q>=0.95)')
    plt.ylabel('Volume')
    plt.xlabel('Timestamp (UTC)')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
else:
    print('No hay volumen > 0 en el dataset (o está todo a 0).')


In [ ]:
# Conclusions (auto-generated, lightweight)
# Nota: estas conclusiones son heurísticas basadas en conteos/transiciones.

summary_lines = []
summary_lines.append('Ventana temporal: ' + str(finviz_df['timestamp'].min()) + ' -> ' + str(finviz_df['timestamp'].max()))
summary_lines.append('Categorias únicas: ' + str(finviz_df['category'].nunique()))
summary_lines.append('Tickers únicos: ' + str(finviz_df['ticker'].nunique()))

if len(trans_counts) > 0:
    top_trans = trans_counts.head(5).copy()
    top_trans['pair'] = top_trans['from_category'] + ' -> ' + top_trans['to_category']
    summary_lines.append('Top transiciones: ' + ', '.join(top_trans['pair'].tolist()))

if len(spikes) > 0:
    top_sp = spikes.sort_values('volume', ascending=False).head(3)
    top_sp_str = []
    for _, r in top_sp.iterrows():
        top_sp_str.append(str(r['ticker']) + ' (' + str(int(r['volume'])) + ')')
    summary_lines.append('Top picos de volumen: ' + ', '.join(top_sp_str))

print('
'.join(summary_lines))
